# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset using the `mlcroissant` library, referencing the Croissant schema entities by their `@id` fields as per best practices.

### Dataset Source
The dataset source is provided via this Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is available
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and reference the main entities using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

md = dataset.metadata
print("Dataset Title:", md.name)
print("Description:", md.description)
print("Version:", md.version)


## 2. Data Overview

Examine available record sets, fields, and their `@id`s. All entities are referenced by `@id` according to the dataset schema.

In [ ]:
# List all record sets and their IDs
record_sets = [rs['@id'] for rs in md['recordSet']] if hasattr(md, '__getitem__') and 'recordSet' in md else []

if not record_sets:
    # As per Croissant v1.1.3, prefer the official API
    record_sets = [r['@id'] for r in dataset._get_canonical_json()['recordSet']]

print("Record Sets (@id):")
for i, rs_id in enumerate(record_sets):
    print(f"{i+1}. {rs_id}")

# For each record set, print fields and columns
for rs_id in record_sets:
    rs_obj = next((r for r in dataset._get_canonical_json()['recordSet'] if r['@id'] == rs_id), None)
    print(f"\nRecord Set '@id': {rs_id}")
    if rs_obj:
        fields = rs_obj.get('field', [])
        if not isinstance(fields, list):
            fields = [fields]
        print("  Fields and Columns (@id):")
        for f in fields:
            if isinstance(f, dict):
                f_id = f.get('@id')
                col_ids = f.get('column', None)
            else:
                f_id = f
                col_ids = None
            print(f"    - Field @id: {f_id}")
            if col_ids:
                if isinstance(col_ids, list):
                    for col in col_ids:
                        print(f"       - Column @id: {col}")
                else:
                    print(f"       - Column @id: {col_ids}")
    else:
        print("  [Unable to resolve fields]")

## 3. Data Extraction

Load data from a specified record set into a pandas DataFrame for analysis. Always specify record sets and fields by their `@id`.

In [ ]:
# Choose one record set to extract (example: the main experimental data)
# In this dataset, record set ID is likely 'https://api.app.sen.science/frontiers/7154140/51208a60-7820-430e-9645-c9a740915fea'
# However, fetch the first recordSet listed to demonstrate generic logic
selected_record_set_id = record_sets[0] if record_sets else None

# Extract all data from the chosen record set
dataframes = {}
if selected_record_set_id:
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    dataframes[selected_record_set_id] = df
    print(f"Fields in Record Set '{selected_record_set_id}':")
    print(df.columns.tolist())
    display(df.head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing and exploration steps using field `@id` references. 

Below we:
- Select a numeric field using its `@id`
- Filter records by a threshold
- Normalize the field
- Optionally, group data by another field

In [ ]:
# Example: select 'coverage_percent' as a numeric field if present
# Use its proper @id if known, otherwise, use the column name directly
df = dataframes[selected_record_set_id]

# View column names for guidance
print("Columns available for EDA:", df.columns.tolist())

# Attempt to guess a numeric field (e.g., 'coverage_percent', 'peptide_count', 'MW')
possible_numeric_fields = [
    c for c in df.columns 
    if c.lower() in ['mw', 'molecular_weight', 'peptide_count', 'coverage_percent'] or pd.api.types.is_numeric_dtype(df[c])
]

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")

    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold} (showing first 5):")
    display(filtered_df.head())

    # Normalize
    normalized_column = f"{numeric_field}_normalized"
    filtered_df[normalized_column] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} (first 5):")
    display(filtered_df[[numeric_field, normalized_column]].head())

    # Try grouping by a categorical field (e.g., 'EV_sample', 'condition')
    group_fields = [c for c in df.columns if ('sample' in c.lower() or 'condition' in c.lower()) and df[c].dtype == 'O']
    if group_fields:
        group_field = group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field} (first 5):")
        display(grouped_df.head())
    else:
        print("No categorical group fields (e.g. 'sample', 'condition') found.")
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization

Visualize the filtered numeric field's distribution and any observed grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if possible_numeric_fields:
    plt.figure(figsize=(8,5))
    sns.histplot(filtered_df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field} (> {threshold})")
    plt.xlabel(numeric_field)
    plt.show()

    if group_fields:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

- This notebook demonstrated dynamic exploration of the FAIR2 dataset via Croissant schema.
- All core dataset entities (record sets, fields) are referenced by their `@id`.
- We loaded experimental data, filtered and normalized a numeric field, and visualized distributions and group differences.

Further analyses can leverage the full richness of the schema by addressing additional record sets or combining fields, using `mlcroissant`'s API and the power of pandas and visualization libraries.